# ゼロから作るショアのアルゴリズム 第1回: 量子加算器

このコレクションの[別のチュートリアル](ja/400_shor.ipynb)では、すでにショアのアルゴリズムの*理論*を最初から最後まで解説しています。
ここから始まるのは、その理論を**実際に動く実装**として、算術演算の部品を1つずつ積み上げていく連載です: 量子加算器 → 剰余加算器 → 制御剰余乗算器 → べき剰余 → アルゴリズム全体。各回は準備ができ次第、順次追加していく予定です。

今回は、最初の最も基本的な部品である「2つの2進数を足し算する量子回路」を扱います。

参考文献: V. Vedral, A. Barenco, A. Ekert, ["Quantum Networks for Elementary Arithmetic Operations"](https://arxiv.org/abs/quant-ph/9511018) (1995)

## なぜ加算器が必要か

ショアのアルゴリズムの構造をおさらいしましょう(完全な導出は[400_shor.ipynb](ja/400_shor.ipynb)を参照してください)。量子部分がやっているのは、ある $x$ の $N$ を法とした*位数* $r$ (つまり $x^r \equiv 1 \pmod N$ となる最小の $r$)を見つけることです。$r$ を見つけることは、多数の $j$ について重ね合わせ状態のまま一斉に $x^j \bmod N$ を計算すること、すなわち**べき剰余(modular exponentiation)**に帰着します。べき剰余は**制御剰余乗算**の繰り返しから構成され、制御剰余乗算はさらに**剰余加算**の繰り返しから構成されます。つまり、この積み上げの一番下にあるのは「量子コンピュータでどうやって2つの数を足すか」という問題です。

## 古典ゲートから量子ゲートへ

以下の3つの量子ゲートがあれば、任意の古典的(可逆)論理を再現できます。

| 古典演算 | 量子ゲート |
|:---|:---|
| NOT | `X` |
| XOR | `CX` (CNOT) |
| AND | `CCX` (トフォリ) |

`CX[a, b]` は量子ビット `a` が `1` のとき量子ビット `b` を反転させます — これはまさに `b := b XOR a` です。`CCX[a, b, c]` は `a` と `b` が*両方とも* `1` のときだけ `c` を反転させます — `c` が `0` から始まる場合、これは `c := c OR (a AND b)` になります。通常の2進数の1桁の足し算は、まさにこの2つの演算だけから構成されます: 桁上げ出力ビットは入力(と繰り上がってきた桁上げ)のANDであり、和ビットは入力(と繰り上がってきた桁上げ)のXORです。

## CARRYとSUM

ある1桁について、入ってくる桁上げを $c$、加数を $a$、$b$、出ていく桁上げを $c_{next}$ とすると、

```
CARRY: c,a,b,c_next -> c,a,b, c_next XOR (a+b+c の桁上げ)
SUM:   c,a,b         -> c,a, (a XOR b XOR c)
```

最下位ビットから順に `CARRY` を連鎖させれば、すべての桁上げビットが得られます。その後 `SUM` を使って、各桁上げを対応する和ビットに変換します。量子ゲートは可逆でなければならないため、`CARRY` と `SUM` はどちらも `CCX`/`CX` だけで構成された回路であり、自動的に可逆になっています。このあと `CARRY` の逆演算(`CARRY_INV`)が必要になります。

In [1]:
!pip install git+https://github.com/blueqat/blueqatSDK

/Users/yuichirominato/.pyenv/shims/pip: line 8: /opt/homebrew/opt/pyenv/bin/pyenv: No such file or directory


In [2]:
from blueqat import Circuit

def carry(c, a, b, c_next):
    """CCX[a,b,c_next].CX[a,b].CCX[c,b,c_next] -- propagates the carry out of bit position a+b+c into c_next."""
    circ = Circuit()
    circ.ccx[a, b, c_next]
    circ.cx[a, b]
    circ.ccx[c, b, c_next]
    return circ

def carry_inv(c, a, b, c_next):
    """The inverse of carry(): same three (self-inverse) gates, applied in reverse order."""
    circ = Circuit()
    circ.ccx[c, b, c_next]
    circ.cx[a, b]
    circ.ccx[a, b, c_next]
    return circ

def sum_gate(c, a, b):
    """CX[a,b].CX[c,b] -- turns b into a XOR b XOR c (the sum bit), leaving a and c unchanged."""
    circ = Circuit()
    circ.cx[a, b]
    circ.cx[c, b]
    return circ

## 加算器全体の組み立て

$n$ビットの数 $a$ と $b$ を足すには $3n+1$ 個の量子ビットが必要で、これを3つのレジスタに分けます。

- `c[0..n-1]`: 桁上げレジスタ。$\ket{0}$ から始まる
- `a[0..n-1]`: 1つ目の加数
- `b[0..n]`: 2つ目の加数。**入力より1ビット広い** — 和 $a+b$ は1桁増える可能性があり、最終的な答えはここに格納される

回路は、下位ビットから `carry` を連鎖させて桁上げを一番上(余分なビットである `b[n]`)まで伝搬させ、最上位ビットに `sum_gate` を適用します。その後、下に向かって戻りながら各ビットで `carry_inv` と `sum_gate` を適用し、残りの和ビットをすべて計算すると同時に、桁上げレジスタを再利用または破棄できるようにきれいに**アンコンピュート**(全て $\ket{0}$ に戻す)します。

In [3]:
def build_adder(input_a, input_b, n):
    """n-bit ripple-carry adder. Returns (circuit, b_register_qubit_indices)."""
    num_qubits = 3 * n + 1
    circ = Circuit(num_qubits)

    c = list(range(n))
    a = list(range(n, 2 * n))
    b = list(range(2 * n, 3 * n + 1))

    # Encode the two inputs (c stays at |0...0>)
    for i in range(n):
        if (input_a >> i) & 1:
            circ.x[a[i]]
        if (input_b >> i) & 1:
            circ.x[b[i]]

    # Ripple the carry up
    for i in range(n - 1):
        circ += carry(c[i], a[i], b[i], c[i + 1])
    circ += carry(c[n - 1], a[n - 1], b[n - 1], b[n])

    # Top sum bit
    circ.cx[a[n - 1], b[n - 1]]
    circ += sum_gate(c[n - 1], a[n - 1], b[n - 1])

    # Walk back down: uncompute the carries, filling in the remaining sum bits
    for i in range(n - 2, -1, -1):
        circ += carry_inv(c[i], a[i], b[i], c[i + 1])
        circ += sum_gate(c[i], a[i], b[i])

    return circ, b

## 答えの読み出し方

**ビット順序についての注意**: このSDKの `.run(shots=...)` は、量子ビット0を結果文字列の*右端*の文字として返します(標準的な2進数表記の慣習)。したがって、長さ `N` の結果文字列から量子ビット `q` の値を読み出すには `state[N - 1 - q]` を参照します。`b[0]` は和の最下位ビットなので、`b` レジスタを `b[0]`(最下位)から `b[n]`(最上位)まで読み取り、それに応じて整数値を組み立てます。

この回路は `X`/`CX`/`CCX` だけ — つまり重ね合わせを作らず量子ビットの値を反転させるだけのゲート — から構成されているため、決まった入力に対しては常に決まった出力が得られます。したがって `shots=1` だけで毎回正確な答えが得られます。

In [4]:
def run_adder(val_a, val_b):
    n = max(val_a, val_b, 1).bit_length()
    circuit, b = build_adder(val_a, val_b, n)
    state = circuit.m[:].run(shots=1).most_common(1)[0][0]
    total_qubits = len(state)
    bits = [state[total_qubits - 1 - q] for q in b]  # bits[0] = LSB
    return int("".join(reversed(bits)), 2)

In [5]:
for x, y in [(10, 6), (1, 1), (3, 5), (7, 7), (0, 5), (15, 15)]:
    result = run_adder(x, y)
    print(f"{x} + {y} = {result}  (expected {x + y}, {'OK' if result == x + y else 'MISMATCH'})")

10 + 6 = 16  (expected 16, OK)
1 + 1 = 2  (expected 2, OK)
3 + 5 = 8  (expected 8, OK)
7 + 7 = 14  (expected 14, OK)
0 + 5 = 5  (expected 5, OK)
15 + 15 = 30  (expected 30, OK)


6つのケースすべてが通常の整数の足し算と完全に一致しました。純粋な可逆論理回路であることを考えれば、これは期待通りの結果です。

## 次回予告

この加算器は、積み上げの最も内側にある部品です。次のチュートリアルでは、この上に**剰余**加算($a+b \bmod N$)を実装します — べき剰余のすべての中間値は $[0, N)$ の範囲に収まっている必要があるため、これがショアのアルゴリズムで実際に必要とされる演算です。